# generate atrophy maps by taking the mean of the PBS and dividng by the STD

In [1]:
# generate the mean and std of the PBS group

In [ ]:
PBS=('sub-hM835603942971011M8'
    'sub-hM8362824086011084M8'
    'sub-hM836223979691084M1'
    'sub-hM8362224086051084M2'
    'sub-hM8362234086021084M3')


PFF=('sub-hM8362334086041083M6'
    'sub-hM836323993941035M4'
    'sub-hM8363224086001085M5'
    'sub-hM836373993921113M9'
    'sub-hM8362324086031084M5')

In [4]:
full_jac_dir = "/Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/dbm_output_original_resolution/dbm/jacobian/full"
PBS_1 = f'{full_jac_dir}/sub-hM835603942971011M8_ses-01_run-1_T1w.nii.gz'
PBS_2 = f'{full_jac_dir}/sub-hM8362824086011084M8_ses-01_run-1_T1w.nii.gz'
PBS_3 = f'{full_jac_dir}/sub-hM836223979691084M1_ses-01_run-1_T1w.nii.gz'
PBS_4 = f'{full_jac_dir}/sub-hM8362224086051084M2_ses-01_run-1_T1w.nii.gz'
PBS_5 = f'{full_jac_dir}/sub-hM8362234086021084M3_ses-01_run-1_T1w.nii.gz'

In [5]:
# merge the PBS images into a 4D image
!fslmerge -t $full_jac_dir/PBS_4D.nii.gz $PBS_1 $PBS_2 $PBS_3 $PBS_4 $PBS_5

In [6]:
# take the mean of the PBS group and std
!fslmaths $full_jac_dir/PBS_4D.nii.gz -Tmean $full_jac_dir/PBS_mean.nii.gz
!fslmaths $full_jac_dir/PBS_4D.nii.gz -Tstd $full_jac_dir/PBS_std.nii.gz

In [13]:
%%bash
# for each PFF image, subtract the mean of the PBS group and divide by the std of the PBS group

full_jac_dir="/Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/dbm_output_original_resolution/dbm/jacobian/full"

PFF=('sub-hM8362334086041083M6'
    'sub-hM836323993941035M4'
    'sub-hM8363224086001085M5'
    'sub-hM836373993921113M9'
    'sub-hM8362324086031084M5')
for pff in "${PFF[@]}"; do
    echo "Processing $pff"
    fslmaths $full_jac_dir/${pff}_ses-01_run-1_T1w.nii.gz \
    -sub $full_jac_dir/PBS_mean.nii.gz \
    -div $full_jac_dir/PBS_std.nii.gz $full_jac_dir/${pff}_atrophy_map.nii.gz

done

Processing sub-hM8362334086041083M6
Processing sub-hM836323993941035M4
Processing sub-hM8363224086001085M5
Processing sub-hM836373993921113M9
Processing sub-hM8362324086031084M5


In [14]:
ABA_in_avg_space = "/Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/dbm_output_original_resolution/final-target/ABA_51_bilateral_resampled.nii.gz"

# ========================  =========================

## get average values of the atrophy maps in each ROI


In [18]:
import numpy as np
import pandas as pd
import nibabel as nib
import os

DEFAULT_ATLAS     = '/Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/dbm_output_original_resolution/final-target/ABA_51_bilateral_resampled.nii.gz'
DEFAULT_ATLAS_CSV = '/Users/aeed/Documents/Work/RABIES_templates/ROIs_51_bilateral_interleaved_RABIES_labels.csv'

def extract_roi_means(
    img_path,
    output_dir,
    atlas_path=DEFAULT_ATLAS,
    atlas_csv=DEFAULT_ATLAS_CSV,
):
    # Load images
    img_data   = nib.load(img_path).get_fdata()
    atlas_data = nib.load(atlas_path).get_fdata()
    labels_df  = pd.read_csv(atlas_csv)

    # Compute average voxel value per ROI
    means = []
    for _, row in labels_df.iterrows():
        mask     = atlas_data == row['label_value']
        mean_val = img_data[mask].mean() if mask.any() else np.nan
        means.append(mean_val)

    # Build output dataframe
    out_df = pd.DataFrame({
        'index':            labels_df['label_value'].values,
        'name':             labels_df['name'].values,
        'mean_voxel_value': means,
    })

    # Save
    os.makedirs(output_dir, exist_ok=True)
    img_stem  = os.path.basename(img_path).replace('.nii.gz', '').replace('.nii', '')
    out_path  = os.path.join(output_dir, f'{img_stem}_roi_means.tsv')
    out_df.to_csv(out_path, sep='\t', index=False)

    return out_path

In [19]:
# loop over all the pff atrophy maps and extract the mean values in each ROI
full_jac_dir = "/Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/dbm_output_original_resolution/dbm/jacobian/full"
PFF=['sub-hM8362334086041083M6',
    'sub-hM836323993941035M4',
    'sub-hM8363224086001085M5',
    'sub-hM836373993921113M9',
    'sub-hM8362324086031084M5']
for pff in PFF:
    img_path = f"{full_jac_dir}/{pff}_atrophy_map.nii.gz"
    extract_roi_means(img_path, full_jac_dir)